# Prepare Training Data for Mask R-CNN
This Jupyter notebook should be used to prepare data for training torchvision Mask R-CNN model. The mask annotations (for different instances of objects) are  used to create the training data in the following steps:
- Read the mask annotations (by the annotators) and parse them.  
- Crop the large images to smaller sub-images to keep the object sizes large enough for the model to be able to detect them. 
- Save the resulting sub-images and the annotations to disk to be read during the training. 

In [ ]:
# load required libraries
import sys
sys.path.append('../utils')
from pairing_utils import iou_batch, iou_mask_pair
# data model - Reading annotated data (by annotators)
# the data model class below returns masks as well
from json_parser import CellMaskDataset, MaskDatasetFromMultiAnnotations, optimize_crop, crop_and_block, show_sample

import os
from PIL import Image
from IPython.display import display
import numpy as np
import shutil
import cv2
import pandas as pd
import pickle
import pycocotools
from pycocotools import mask as coco_mask_util
from typing import List, Union, Dict, Final, Tuple

### Train and test sets
The mapping between the class IDs and the class names should have the same class names used in the annotations as the values. 

In [ ]:
# this folder will include all images (train or test)
# if an image is missing, it will get downloaded from the url specified in the annotation file
# the image names should be the same as the annotation file names

IMAGES_PATH = '/home/cellareye/Cellanome/Data/Images'

# TRAIN_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/analysis-images-batch-2-121422-not-reviewed/train'
# TEST_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/analysis-images-batch-2-121422-not-reviewed/test'

# TRAIN_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/normalised-12212022-tregs-beads-cages-bb2-not-reviewed/train'
# TEST_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/normalised-12212022-tregs-beads-cages-bb2-not-reviewed/test'

# LABEL_MAP = {1: 'Cell', 2: 'Bead', 3: 'cages'}
# REVERSE_LABEL_MAP = {value: key for key, value in LABEL_MAP.items()}


TRAIN_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/microscope-images-batch-1-091922-not-reviewed/train'
TEST_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/microscope-images-batch-1-091922-not-reviewed/test'

RESIZED_BB_IMAGE_SIZE = 2720

LABEL_MAP =  {1: 'Cell', 2: 'Bead'}
# map  'dying/dead cells' to 'Cell' (with class_id = 1) and ignore 'Cluster'
REVERSE_LABEL_MAP = {'Cell': 1, 'dying/dead cells': 1, 'Bead': 2}

In [ ]:
# use color depth = 8 for images shared with annotators; these are already processed images
# we do not do any resizing before cropping the images later into smaller sub-images
# here is the size distribution of the set for the caging model
# (1600, 2000): 1201, (1944, 2592): 41, (2208, 2758): 69, (2208, 2756): 15
# here is the size distribution of the set for the analysis model (first batch)
# (1600, 2000): 1377, (4512, 4512): 313
# here is the size distribution of the set for the analysis model (second batch)
# (4512, 4512): 487

# the objects' bounding boxes are only used for optimizing the crop sizes in this notebook and are
# not included in the parsed data (masks)
# hence, there is no reason to expand them during the parsing (percentage_to_expand_bbox_boundaries is 
# set to zero)

train_annotation_files: List[str] = os.listdir(TRAIN_ANNOTS_PATH)
train_annotation_files = ['.'.join(file.strip().split('.')[:-1]) for file in train_annotation_files]

test_annotation_files: List[str] = os.listdir(TEST_ANNOTS_PATH)
test_annotation_files = ['.'.join(file.strip().split('.')[:-1]) for file in test_annotation_files]

train_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=TRAIN_ANNOTS_PATH, 
                                annotations=train_annotation_files,
                                labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                                percentage_to_expand_bbox_boundaries = 0.0, 
                                color_depth=8, 
                                min_object_diameter = 6.0,
                                scale_factor_dict={(2000, 1600): 1.11111111}, 
                                max_larger_side = RESIZED_BB_IMAGE_SIZE, max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                                normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)


test_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=TEST_ANNOTS_PATH, 
                               annotations=test_annotation_files ,
                               labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                               percentage_to_expand_bbox_boundaries = 0.0, 
                               color_depth=8, 
                               min_object_diameter = 6.0,
                               scale_factor_dict={(2000, 1600): 1.11111111}, 
                               max_larger_side = RESIZED_BB_IMAGE_SIZE, max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                               normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)

In [ ]:
IMAGES_PATH = '/home/cellareye/Cellanome/Data/nuclei_bf_images'
ANNOTS_PATH = '/home/cellareye/Cellanome/Data/nuclei_bf_annotations'

# the location of two test.txt and train.txt files listing the names (excluding extensions)
# of the images assigned to each set
TRAIN_TEST_SPLIT_FOLDERS = ['/home/cellareye/Cellanome/Data/new_datasets/230607_IMR90_training_dataset_1_nuclei_bf', 
                            '/home/cellareye/Cellanome/Data/new_datasets/230622_IMR90_training_dataset_3_nuclei_bf']

test_annotation_files: List[str] = []
train_annotation_files: List[str] = []
for folder in TRAIN_TEST_SPLIT_FOLDERS:
    with open(os.path.join(folder, 'test.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        test_annotation_files += filenames

    with open(os.path.join(folder, 'train.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        train_annotation_files += filenames

RESIZED_BB_IMAGE_SIZE = 2720
LABEL_MAP =  {1: 'nucleus'}
REVERSE_LABEL_MAP = {'nucleus': 1}

# datasets
train_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=ANNOTS_PATH, 
                                annotations=train_annotation_files,
                                labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                                percentage_to_expand_bbox_boundaries = 0.0, 
                                color_depth=8, 
                                min_object_diameter = 6.0,
                                scale_factor_dict={(2000, 1600): 1.11111111}, 
                                max_larger_side = RESIZED_BB_IMAGE_SIZE, max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                                normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)


test_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=ANNOTS_PATH, 
                               annotations=test_annotation_files ,
                               labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                               percentage_to_expand_bbox_boundaries = 0.0, 
                               color_depth=8, 
                               min_object_diameter = 6.0,
                               scale_factor_dict={(2000, 1600): 1.11111111}, 
                               max_larger_side = RESIZED_BB_IMAGE_SIZE, max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                               normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)

In [ ]:
IMAGES_PATH = '/home/cellareye/Cellanome/Data/cytoplasm_nucleus_cage_combined/images'
ANNOTS_PATH = '/home/cellareye/Cellanome/Data/cytoplasm_nucleus_cage_combined/annotations'


test_files: List[str] = []
train_files: List[str] = []

with open(os.path.join('/home/cellareye/Cellanome/Data/cytoplasm_nucleus_cage_combined', 'test.txt')) as file:
    filenames = file.readlines()
    filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
    test_files += filenames

with open(os.path.join('/home/cellareye/Cellanome/Data/cytoplasm_nucleus_cage_combined', 'train.txt')) as file:
    filenames = file.readlines()
    filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
    train_files += filenames

RESIZED_BB_IMAGE_SIZE = 2720
LABEL_MAP =  {5: 'cytoplasm', 4: 'nucleus', 3: 'cages'}
REVERSE_LABEL_MAP = {value: key for key, value in LABEL_MAP.items()}


# datasets
train_dataset = MaskDatasetFromMultiAnnotations(images_path=IMAGES_PATH, 
                                                annotations_paths=[ANNOTS_PATH, ANNOTS_PATH, ANNOTS_PATH],  
                                                common_names_to_use=train_files,
                                                annotation_files_exts=['cyto', 'nucl', 'cage'], 
                                                labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                                                percentage_to_expand_bbox_boundaries=0.0, 
                                                color_depth=8, 
                                                min_object_diameter=6.0,
                                                scale_factor_dict={(2000, 1600): 1.11111111}, 
                                                max_larger_side = RESIZED_BB_IMAGE_SIZE, 
                                                max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                                                normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)


test_dataset = MaskDatasetFromMultiAnnotations(images_path=IMAGES_PATH, 
                                               annotations_paths=[ANNOTS_PATH, ANNOTS_PATH, ANNOTS_PATH],  
                                               common_names_to_use=test_files,
                                               annotation_files_exts=['cyto', 'nucl', 'cage'], 
                                               labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                                               percentage_to_expand_bbox_boundaries=0.0, 
                                               color_depth=8, 
                                               min_object_diameter=6.0,
                                               scale_factor_dict={(2000, 1600): 1.11111111}, 
                                               max_larger_side = RESIZED_BB_IMAGE_SIZE, 
                                               max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                                               normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)


In [ ]:
IMAGES_PATH = '/home/cellareye/Cellanome/Data/cytoplasm_nucleus_cage_overlaid/images'
ANNOTS_PATH = '/home/cellareye/Cellanome/Data/cytoplasm_nucleus_cage_overlaid/annotations'


test_files: List[str] = []
train_files: List[str] = []

with open(os.path.join('/home/cellareye/Cellanome/Data/cytoplasm_nucleus_cage_overlaid', 'test.txt')) as file:
    filenames = file.readlines()
    filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
    test_files += filenames

with open(os.path.join('/home/cellareye/Cellanome/Data/cytoplasm_nucleus_cage_overlaid', 'train.txt')) as file:
    filenames = file.readlines()
    filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
    train_files += filenames

RESIZED_BB_IMAGE_SIZE = 2720
LABEL_MAP =  {5: 'cell-adhered', 4: 'nucleus', 3: 'cages', 2: 'bead', 1:'cell'}
REVERSE_LABEL_MAP = {value: key for key, value in LABEL_MAP.items()}


# datasets
train_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=ANNOTS_PATH, 
                                annotations=train_files,
                                labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                                percentage_to_expand_bbox_boundaries = 0.0, 
                                color_depth=8, 
                                min_object_diameter = 6.0,
                                scale_factor_dict={(2000, 1600): 1.11111111}, 
                                max_larger_side = RESIZED_BB_IMAGE_SIZE, max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                                normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)


test_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=ANNOTS_PATH, 
                               annotations=test_files ,
                               labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                               percentage_to_expand_bbox_boundaries = 0.0, 
                               color_depth=8, 
                               min_object_diameter = 6.0,
                               scale_factor_dict={(2000, 1600): 1.11111111}, 
                               max_larger_side = RESIZED_BB_IMAGE_SIZE, max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                               normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)

In [ ]:
IMAGES_PATH = '/home/cellareye/Cellanome/Data/Images'

# TRAIN_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/analysis-images-batch-2-121422-not-reviewed/train'
# TEST_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/analysis-images-batch-2-121422-not-reviewed/test'

TRAIN_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/normalised-12212022-tregs-beads-cages-bb2-not-reviewed/train'
TEST_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/normalised-12212022-tregs-beads-cages-bb2-not-reviewed/test'

RESIZED_BB_IMAGE_SIZE = 2720
LABEL_MAP = {2: 'cages'}
REVERSE_LABEL_MAP = {'cages': 2}

train_annotation_files: List[str] = os.listdir(TRAIN_ANNOTS_PATH)
train_annotation_files = ['.'.join(file.strip().split('.')[:-1]) for file in train_annotation_files]

test_annotation_files: List[str] = os.listdir(TEST_ANNOTS_PATH)
test_annotation_files = ['.'.join(file.strip().split('.')[:-1]) for file in test_annotation_files]

train_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=TRAIN_ANNOTS_PATH, 
                                annotations=train_annotation_files,
                                labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                                percentage_to_expand_bbox_boundaries = 0.0, 
                                color_depth=8, 
                                min_object_diameter = 6.0,
                                scale_factor_dict={(2000, 1600): 1.11111111}, 
                                max_larger_side = RESIZED_BB_IMAGE_SIZE, max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                                normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)


test_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=TEST_ANNOTS_PATH, 
                               annotations=test_annotation_files ,
                               labels_of_interest=list(REVERSE_LABEL_MAP.keys()), 
                               percentage_to_expand_bbox_boundaries = 0.0, 
                               color_depth=8, 
                               min_object_diameter = 6.0,
                               scale_factor_dict={(2000, 1600): 1.11111111}, 
                               max_larger_side = RESIZED_BB_IMAGE_SIZE, max_smaller_side = RESIZED_BB_IMAGE_SIZE,
                               normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)

### Clean up annotations and enforce a one-to-one mapping between child-parent pairs of objects 
In some scenarios, there is a hierarchy between objects of different classes. For example, the mask of an object from class `nucleus` should always be inside the mask of another object of class `cytoplasm`. Furthermore, some times this mapping has to be one-to-one. A `cytoplasm` object can only have one `nucleus`, and a `nucleus` object should belong to only one `cytoplasm`. In many annotated images, different objects of class `cytoplasm` cannot be accurately differentiated and multiple objects are combined (hence have multiple nuclei). We remove these invalid annotations below. 

In [ ]:
def enforce_one_to_one_mapping(data_sample: dict, child_parent_map: Dict[int, int]) -> dict:
    
    # make sure the index in the DataFrame is the same as the row index
    annotations: pd.DataFrame  = data_sample['annotations'].reset_index(drop=True).copy()
    
    # the indexes of invalid objects from the child and parent classes
    # objects from the child classes (with class IDs specified by the keys in child_parent_map)
    # that have no objects from the mapped parent class including them, or have multiple objects from 
    # the mapped parent class covering them (not expected)
    invalid_child_idxs: List[int] = []
    # objects from the parent classes (with class IDs specified by the values in child_parent_map)
    # that have no objects from the mapped child class belonging to them, or cover multiple objects from 
    # the mapped child class 
    invalid_parent_idxs: List[int] = []
        
    # go over the child-parent pairs for which one-to-one mapping should be enforced
    for child_class_id, parent_class_id in child_parent_map.items():
    
    
        child_df: pd.DataFrame = annotations[annotations['label'] == child_class_id]
        parent_df: pd.DataFrame = annotations[annotations['label'] == parent_class_id]
        
        # indexes
        child_idxs: np.ndarray = child_df.index.values
        parent_idxs: np.ndarray = parent_df.index.values
            
        if len(child_idxs) == 0:
            if len(parent_idxs) > 0:
                invalid_parent_idxs += parent_idxs.tolist()    
            continue
        elif len(parent_idxs) == 0:
            invalid_child_idxs += child_idxs.tolist()
            continue    
        
        # bounding boxes
        child_boxes: np.ndarray = child_df[['xtl', 'ytl', 'xbr', 'ybr']].values
        parent_boxes: np.ndarray = parent_df[['xtl', 'ytl', 'xbr', 'ybr']].values

        # masks
        child_masks: List[np.ndarray] = [data_sample['masks'][idx] for idx in child_idxs]
        parent_masks: List[np.ndarray] = [data_sample['masks'][idx] for idx in parent_idxs]

        # the IoU matrix for the bounding boxes, element (i, j) in the matrix below is the 
        # IoU between the bounding box of the child object i (with index child_idxs[i] in the 
        # data_sample['annotations'] and data_sample['masks']) and the bounding box of the 
        # parent object j (with index parent_idxs[j])
        iou_box_matrix: np.ndarray = iou_batch(child_boxes, parent_boxes)
    
        overlapping_child_idxs, overlapping_parent_idxs = np.where(iou_box_matrix > 0)
    
        # the IoU matrix for the masks, for efficiency reasons, it is only calculated
        # for pairs with box IoU > 0
        iou_mask_matrix: np.ndarray = np.zeros((len(child_idxs), len(parent_idxs)))
    
        for (i, j) in zip(overlapping_child_idxs, overlapping_parent_idxs):        
            iou_mask_matrix[i, j] = iou_mask_pair(child_boxes[i], child_masks[i], 
                                                  parent_boxes[j], parent_masks[j])

        iou_mask_matrix[iou_mask_matrix > 0] = 1
        
        # the (columns) index of parent objects without an object from the child class or 
        # with multiple objects from the child class 
        invalid_col_idxs: np.ndarray = np.where(np.sum(iou_mask_matrix, axis=0) != 1)[0]
        # only keep objects from the parent class with one child, before finding invalid 
        # objects of class child
        valid_col_idxs: np.ndarray = np.array([idx for idx in range(len(parent_idxs)) 
                                               if idx not in invalid_col_idxs])
            
        if len(valid_col_idxs) == 0:
            invalid_row_idxs: np.ndarray = np.array([idx for idx in range(len(child_idxs))])
        else:
            # the (row) index of child objects without an object from the parent class or 
            # with multiple objects from the parent class 
            invalid_row_idxs: np.ndarray = np.where(np.sum(iou_mask_matrix[:, valid_col_idxs], axis=1) != 1)[0]    
            
        
        invalid_child_idxs += child_idxs[invalid_row_idxs].tolist()
        invalid_parent_idxs += parent_idxs[invalid_col_idxs].tolist()
    
    # black out invalid objects in the image, and remove them from the annotations and masks
    img: np.ndarray = data_sample['image'].copy()
    img_mask: np.ndarray = np.zeros(img.shape[:2], np.uint8)
    blacked_out_mask: np.ndarray = np.ones(img.shape[:2], np.uint8)

    valid_idxs: List[int] = []
    for idx, row in annotations.iterrows():
        xtl, ytl, xbr, ybr = row[['xtl', 'ytl', 'xbr', 'ybr']].values
        if idx in (invalid_child_idxs + invalid_parent_idxs):
            blacked_out_mask[ytl:ybr, xtl:xbr] = 0
        else:
            img_mask[ytl:ybr, xtl:xbr] = 1
            valid_idxs.append(idx)
    out: dict = {}
    out['name'] = data_sample['name']
    out['image']: np.ndarray = cv2.bitwise_or(blacked_out_mask, img_mask) * img
    out['annotations']: pd.DataFrame = annotations.loc[valid_idxs].reset_index(drop=True).copy()
    out['masks']: List[np.ndarray] = [data_sample['masks'][idx].copy() for idx in valid_idxs]
        
    return out

### Output folders and resized image dimensions

In [ ]:
## make sure these folders are generated in advance
OUTPUT_BASE_PATH = os.path.join(os.getcwd(), 'data/analysis_data_cell_bead_cage_nucleus_cytoplasm_mix_crop')
TRAIN_IMAGE_FOLDER = 'images/train'
TRAIN_MASK_FOLDER = 'masks/train'
TEST_IMAGE_FOLDER = 'images/test'
TEST_MASK_FOLDER = 'masks/test'
# the minimum object mask area to keep the object in the data
MIN_MASK_AREA = 16
# a dictionary to specify a one to one mapping between the classes 
# None should be configured if not needed
CHILD_PARENT_CLASS_MAP = {REVERSE_LABEL_MAP['nucleus']: REVERSE_LABEL_MAP['cell-adhered']}

# the limit on the larger and smaller sides of the input to Mask R-CNN model
# used for preparing un-cropped training images (whole images)
MASK_RCNN_INPUT_WIDTH = 1024
MASK_RCNN_INPUT_HEIGHT = 800
# the minimum and maximum sizes of sqaure cropped images, used for preparing 
# cropped images (smaller than Mask R-CNN input size specified above) for training
CROPPED_IMAGE_SIZE_MIN = 100
CROPPED_IMAGE_SIZE_MAX = 300
# fixed seed to generate the same sets over multiple runs of the code
# (applicable for cropped imaegd)
SEED = 7
np.random.seed(SEED)

### Generate the train and test data with Mask R-CNN input size resolution
Run the cell below twice, with the train flag set to True and False to generate the train and test data for Mask R-CNN training with un-cropped/full images. This will take the annotated data from the train and test classes and generate cropped sub-images with the annotations, and save them into disk. These are then read for training the model. 

In [ ]:
def prepare_data(train=True, save_masks_in_coco_rle_format=False, crop_id='2', child_parent_map=None):
    
    if train:
        set_desc = 'training'
        image_folder = TRAIN_IMAGE_FOLDER
        mask_folder = TRAIN_MASK_FOLDER
        data_set = train_dataset
    else:
        set_desc = 'testing'
        image_folder = TEST_IMAGE_FOLDER
        mask_folder = TEST_MASK_FOLDER
        data_set = test_dataset

    # keep all the labels in the model label map
    class_ids_of_interest = list(LABEL_MAP.keys())

    num_images = 0
    num_annotations = 0

    # read the image and the annotations, then parse each
    for idx in range(len(data_set)):

        sample = data_set[idx]
        
        if child_parent_map is not None:
            sample = enforce_one_to_one_mapping(data_sample=sample, child_parent_map=child_parent_map)
        
        if len(sample['annotations']) == 0:
            continue
        
        # image size
        image_height, image_width = sample["image"].shape[:2]
        
        if (image_width, image_height) not in [(1800, 1440), (RESIZED_BB_IMAGE_SIZE, RESIZED_BB_IMAGE_SIZE)]:
            print('[WARNING: Incorrect image resolution! Skipping the image ...')
            continue
            
        # the number of overlapping pixels between crops in each dimension
        # this is larger than the largest expected cell size (in fact, 3 times 
        # more than the expected size of the cells in breadboard images)
        
        if crop_id == '2':
            overlap_in_y = 160
        elif crop_id == '3':
            overlap_in_y = 525
        else:
            print(f"[ERROR] Invalid passed crop_id: {crop_id}! 2 or 3 supported for now")
            return
        
        if image_height == 1440 and image_width == 1800:
            overlap_in_x = 248
        else:
            if crop_id == '2':
                overlap_in_x = 176
            elif crop_id == '3':
                overlap_in_x = 600
            
        # the step size for the starting point of each crop in x and y dimension
        crop_start_step_x = MASK_RCNN_INPUT_WIDTH - overlap_in_x
        crop_start_step_y = MASK_RCNN_INPUT_HEIGHT - overlap_in_y


        crop_count = 0
        # overlapping crops
        for x_start in range(0, image_width - overlap_in_x, crop_start_step_x):    
            for y_start in range(0, image_height - overlap_in_y, crop_start_step_y):
                # crop coordinates
                xc_tl = x_start
                yc_tl = y_start
                xc_br = x_start + MASK_RCNN_INPUT_WIDTH
                yc_br = y_start + MASK_RCNN_INPUT_HEIGHT
                # make sure we always crop the image with the given size
                # if we get to the boundaries, extend the crop
                # size inside the image to always get the same size crop
                # this is not really needed, but help with capturing more
                # annotations toward the low/right parts of the image
                if xc_br > image_width:
                    xc_br = image_width 
                    xc_tl = max(0, xc_br - MASK_RCNN_INPUT_WIDTH)
                if yc_br > image_height:
                    yc_br = image_height
                    yc_tl = max(0, yc_br - MASK_RCNN_INPUT_HEIGHT)

                crop_coords = [xc_tl, yc_tl, xc_br, yc_br]

                # optimize the crop
                crop_coords = optimize_crop(sample['annotations'], crop_coords, 
                                            int(overlap_in_x / 3), int(overlap_in_y / 3), 
                                            image_width, image_height, class_ids_of_interest)

                # crop and block the image
                cropped_sample =  crop_and_block(sample, crop_coords, labels_of_interest=class_ids_of_interest, 
                                                 keep_area_threshold=0.33)
                
                # skip empty crops
                if len(cropped_sample['annotations']) < 1:
                    continue

                crop_height, crop_width = cropped_sample["image"].shape[:2]
                
                # cropped image and annotations name, for each image/annotation name use _ crop_count
                img_name = ".".join(cropped_sample["name"].strip().split('.')[:-1])
                # image jpg file
                crp_img_name = img_name + '_crp_' + str(crop_count) + '.jpg'
                
                if save_masks_in_coco_rle_format:
                    # save the masks and annotations as a dictionary (pickle file)
                    # as in the following
                    record = {}
                    record['annotations']: list[dict] = []
                    for obj_id, current_mask in enumerate(cropped_sample["masks"]):
                    
                        # check the area first, make sure to include only large enough objects
                        if current_mask.sum() < MIN_MASK_AREA:
                            continue
                    
                        # object's bounding box
                        box_xtl, box_ytl, box_xbr, box_ybr = \
                        cropped_sample['annotations'].loc[obj_id, ['xtl', 'ytl', 'xbr', 'ybr']].values
                        
                        # object's label
                        label = int(cropped_sample['annotations'].loc[obj_id, 'label'])
                        
                        # we do not expand the mask to the crop's resolution and
                        # only save the mask within the objects bounding box to save space
                        num_annotations += 1
                  
                        annots: dict = {'bbox': [box_xtl, box_ytl, box_xbr, box_ybr],
                                        'category_id' : label,
                                        'segmentation': coco_mask_util.encode(np.asarray(current_mask, 
                                                                                         order="F")),
                                       } 
                        record['annotations'].append(annots)
                    
                    if len(record['annotations']) == 0:
                        # skip this sample
                        continue
                    
                    # save the record dictionary as a pickle file
                    crp_mask_name = img_name + '_crp_' + str(crop_count) + '.pkl'
                    
                    filehandler = open(os.path.join(OUTPUT_BASE_PATH, mask_folder, crp_mask_name), 'wb')
                    pickle.dump(record, filehandler)
                    filehandler.close()
                    
                    
                else:
                    # save the mask annotations in our internal format as below:
                    # the masks are saved as an m x crop_height x crop_width array 
                    # (m masks with the same resolution as the cropped image)
                    # and values as np.uint16 
                    # the i-th object in cropped_sample['annotations'] will have 
                    # mask values eqaul to (i + 1) in the first possible array index j
                    # 0 <= j < m where j is chosen such that this mask will have any overlap
                    # with the objects saved on this slice (j)
                    # in the case of having overlapping objects, one of the objects will
                    # be reported in the next/different array index where it does not have 
                    # any overlap with objects in that mask

                    masks: List[np.array] = [np.zeros((crop_height, crop_width), np.uint16)]
                    labels: np.array = np.zeros((len(cropped_sample["masks"]),),  np.uint8) 
                    
                    for obj_id, current_mask in enumerate(cropped_sample["masks"]):
                    
                        # check the area first, make sure to include only large enough objects
                        if current_mask.sum() < MIN_MASK_AREA:
                            continue
                    
                        box_xtl, box_ytl, box_xbr, box_ybr = \
                        cropped_sample['annotations'].loc[obj_id, ['xtl', 'ytl', 'xbr', 'ybr']].values
                        # expand the object mask to cover the full crop resolution
                        full_res_mask: np.array = np.zeros((crop_height, crop_width), np.uint8)
                        full_res_mask[box_ytl:box_ybr, box_xtl:box_xbr] = 1
                    
                        num_annotations += 1
                        # find the first index with no overlap for the object
                        available_array_index = -1
                        for array_index in range(len(masks)):
                            if masks[array_index][full_res_mask > 0].sum() == 0:
                                available_array_index = array_index
                                break

                        if available_array_index < 0:
                            # add a new mask
                            masks.append(np.zeros((crop_height, crop_width), np.uint16))
                            available_array_index = len(masks) - 1

                        masks[available_array_index][full_res_mask > 0] = obj_id + 1
                        labels[obj_id] = int(cropped_sample['annotations'].loc[obj_id, 'label'])

                    
                    if len(masks) == 0 or masks[0].sum() == 0:
                        # skip this cropped sample if no large enough mask was found
                        continue
                
                    # save the annotations (masks) for this image, 
                    # annotation npz file
                    crp_mask_name = img_name + '_crp_' + str(crop_count) + '.npz'

                    # save the masks and labels arrays
                    np.savez(os.path.join(OUTPUT_BASE_PATH, mask_folder, crp_mask_name), 
                             saved_masks = np.array(masks), saved_labels = labels) 
                
                # save the image in jpg format
                cv2.imwrite(os.path.join(OUTPUT_BASE_PATH, image_folder, crp_img_name), cropped_sample["image"])
                
                # increament the counter for images
                num_images += 1
                # increament the crop counter for this image
                crop_count += 1


    print('Created {} '.format(num_images) + 'images for ' + set_desc)
    print('Created {} '.format(num_annotations) + 'objects for ' + set_desc)

### Generate the train and test cropped images, with the maximum and minimum sizes specified 
Run the cell below twice, with the train flag set to True and False to generate the train and test data for Mask R-CNN training with cropped images. This will take the annotated data from the train and test classes and generate cropped sub-images with the annotations, and save them into disk. These are then read for training the model. 

In [ ]:
def prepare_cropped_data(train=True):
    
    if train:
        set_desc = 'training'
        image_folder = TRAIN_IMAGE_FOLDER
        mask_folder = TRAIN_MASK_FOLDER
        data_set = train_dataset
    else:
        set_desc = 'testing'
        image_folder = TEST_IMAGE_FOLDER
        mask_folder = TEST_MASK_FOLDER
        data_set = test_dataset

    # keep all the labels in the model label map
    class_ids_of_interest = list(LABEL_MAP.keys())

    num_images = 0
    num_annotations = 0
   
    # read the image and the annotations, then parse each
    for idx in range(len(data_set)):

        sample = data_set[idx]
        if len(sample['annotations']) == 0:
            continue
        # image size
        image_height, image_width = sample["image"].shape[:2]
        
        # randomly select a crop size that is used for the whole image
        crop_size: int = int(np.random.rand(1)[0] * (CROPPED_IMAGE_SIZE_MAX - CROPPED_IMAGE_SIZE_MIN) +
                             CROPPED_IMAGE_SIZE_MIN)

        crop_count = 0
        
        # add a slight overlap between the crops
        overlap_in_x = int(0.1 * crop_size)
        overlap_in_y = int(0.1 * crop_size)
        
         # the step size for the starting point of each crop in x and y dimension
        crop_start_step_x = crop_size - overlap_in_x
        crop_start_step_y = crop_size - overlap_in_y
        
        # overlapping crops
        for x_start in range(0, image_width - overlap_in_x, crop_start_step_x):    
            for y_start in range(0, image_height - overlap_in_y, crop_start_step_y):
                # crop coordinates
                xc_tl = x_start
                yc_tl = y_start
                xc_br = x_start + crop_size
                yc_br = y_start + crop_size
                # make sure we always crop the image with the given size
                # if we get to the boundaries, extend the crop
                # size inside the image to always get the same size crop
                # this is not really needed, but help with capturing more
                # annotations toward the low/right parts of the image
                if xc_br > image_width:
                    xc_br = image_width 
                    xc_tl = max(0, xc_br - MASK_RCNN_INPUT_WIDTH)
                if yc_br > image_height:
                    yc_br = image_height
                    yc_tl = max(0, yc_br - MASK_RCNN_INPUT_HEIGHT)


                crop_coords = [xc_tl, yc_tl, xc_br, yc_br]

                # optimize the crop
                crop_coords = optimize_crop(sample['annotations'], crop_coords, 
                                            int(crop_size / 2), int(crop_size / 2), 
                                            image_width, image_height, class_ids_of_interest)

                # crop and block the image
                cropped_sample =  crop_and_block(sample, crop_coords, labels_of_interest=class_ids_of_interest, 
                                                 keep_area_threshold=0.33):
                
                # skip empty crops
                if len(cropped_sample['annotations']) < 1:
                    continue

                crop_height, crop_width = cropped_sample["image"].shape[:2]
                
                # create the mask annotations
                # the masks are saved as m x crop_height x crop_width array 
                # (m masks with the same resolution as the cropped image)
                # and values as np.uint16 
                # the i-th object in cropped_sample['annotations'] will have 
                # mask values eqaul to (i + 1) in the first possible array index j
                # 0 <= j < m where it will not overlap with any previously processes
                # object
                # in case of having overlapping objects, one of the objects will
                # be reported in the next array index where it does not have 
                # any overlap with objects in that mask

                masks: List[np.array] = [np.zeros((crop_height, crop_width), np.uint16)]
                labels: np.array = np.zeros((len(cropped_sample["masks"]),),  np.uint8)    
                
                for obj_id, current_mask in enumerate(cropped_sample["masks"]):
                    
                    # check the area first, make sure to include only large enough objects
                    if current_mask.sum() < MIN_MASK_AREA:
                        continue
                    
                    box_xtl, box_ytl, box_xbr, box_ybr = \
                    cropped_sample['annotations'].loc[obj_id, ['xtl', 'ytl', 'xbr', 'ybr']].values
                    
                    full_res_mask: np.array = np.zeros((crop_height, crop_width), np.uint8)
                    full_res_mask[box_ytl:box_ybr, box_xtl:box_xbr] = 1
                    
                    num_annotations += 1
                    # find the first index with no overlap for the object
                    available_array_index = -1
                    for array_index in range(len(masks)):
                        if masks[array_index][full_res_mask > 0].sum() == 0:
                            available_array_index = array_index
                            break

                    if available_array_index < 0:
                        # add a new mask
                        masks.append(np.zeros((crop_height, crop_width), np.uint16))
                        available_array_index = len(masks) - 1

                    masks[available_array_index][full_res_mask > 0] = obj_id + 1
                    labels[obj_id] = int(cropped_sample['annotations'].loc[obj_id, 'label'])
                    
                if len(masks) == 0 or masks[0].sum() == 0:
                    # skip this sample if no large enough mask was found
                    continue
                
                # save the cropped image and the annotations for this image, for each image name
                # use _ crop_count
                img_name = ".".join(cropped_sample["name"].strip().split('.')[:-1])
                # image jpg file
                crp_img_name = img_name + '_crp_' + str(crop_count) + '.jpg'
                # annotation txt file
                crp_mask_name = img_name + '_crp_' + str(crop_count) + '.npz'
                
                # save the image in jpg format
                cv2.imwrite(os.path.join(OUTPUT_BASE_PATH, image_folder, crp_img_name), cropped_sample["image"])
                # save the masks and labels arrays
                np.savez(os.path.join(OUTPUT_BASE_PATH, mask_folder, crp_mask_name), 
                         saved_masks = np.array(masks), saved_labels = labels) 
                # increament the counter for images
                num_images += 1
                # increament the crop counter for this image
                crop_count += 1


    print('Created {} '.format(num_images) + 'images for ' + set_desc)
    print('Created {} '.format(num_annotations) + 'objects for ' + set_desc)

In [ ]:
prepare_data(train=True, save_masks_in_coco_rle_format=True, crop_id='3', 
             child_parent_map=CHILD_PARENT_CLASS_MAP)

In [ ]:
prepare_data(train=False, save_masks_in_coco_rle_format=True, crop_id='3', 
            child_parent_map=CHILD_PARENT_CLASS_MAP)

In [ ]:
prepare_cropped_data(train=True)

In [ ]:
prepare_cropped_data(train=False)

In [ ]:
# 144, 569, 30
# 1293, 5105, 237 

In [ ]:
dx = []
dy = []
for i in range(len(train_dataset)):
    data = train_dataset[i]
    coors = data['annotations'].loc[data['annotations']['label'] == 2, ['xtl', 'ytl', 'xbr', 'ybr']].values
    dx += list(coors[:, 2] - coors[:, 0])
    dy += list(coors[:, 3] - coors[:, 1])

In [ ]:
np.percentile(dy, 90)

In [ ]:
np.percentile(dx, 90)

In [ ]:
# 90th percentile of cages: 527
# 90th percentile of nuclei: 53-59
# 90th percentile of cytoplasm: 500-660 (meaningless)

# 90th percentile of cages: 227